In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
openai_api_key = os.getenv('OPENAI_API_KEY')

# OpenAI API Key가 없으면 예외(오류)를 발생시킨다.
if not openai_api_key:
    raise ValueError('OpenAI API 키가 없습니다. 한 번 더 확인해주세요!') 


In [2]:
# 라이브러리 불러오기
import bs4
from langchain import hub
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
# 문서 불러오기
# 문서 로드
loader = WebBaseLoader(
    web_paths=("https://news.naver.com/section/101",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("sa_text", "sa_item_SECTION_HEADLINE")
        )
    ),
)
docs = loader.load()
docs


[Document(metadata={'source': 'https://news.naver.com/section/101'}, page_content='\n\n한국남부발전, 한화와 글로벌 LNG 협력 강화\n\n한국남부발전은 14일 서울 중구 더 플라자 호텔에서 한화에어로스페이스, 한화에너지와 ‘글로벌 LNG 협력 강화를 위한 Team KOREA 구축’ 업무협약을 체결했다고 밝혔다. 이번 협약은 최근 한미 관세협상 타결로\n\n\n매일경제\n\n\n\n\n18\n개의 관련뉴스 더보기\n\n\n\n\n\n교보생명, 상반기 순익 5824억 전년비 5.4%↓…“미래 수익성 확보”\n\n교보생명이 올해 상반기 5824억원의 순이익을 기록했다. 전년 대비 5.4% 감소한 수치로 보험영업이익이 감소했기 때문이다. 서울 광화문 소재 교보생명 본사 전경.(사진=교보생명) 14일 교보생명은 ‘2025년 상반\n\n\n이데일리\n\n\n\n\n13\n개의 관련뉴스 더보기\n\n\n\n\n\n강릉·속초·경주·익산 ‘1주택자 세컨드홈 특례 지역’ 추가\n\n정부가 1주택자가 비수도권에 집을 한 채 더 살 때 각종 세제 혜택을 받는 ‘세컨드홈’ 특례 대상 지역에 강릉·속초·익산·경주 등 9개 지역을 추가한다. 극심한 침체에서 벗어나지 못하고 있는 지방 건설 경기를 활성화\n\n\n동아일보\n\n\n\n\n70\n개의 관련뉴스 더보기\n\n\n\n\n\n혼조세 보이는 서울 집값...커진 상승폭 다시 줄었다\n\n부동산원 주간 아파트 동향 서울 0.14%→0.10% 축소 서울 아파트값 상승폭이 일주일 사이 다시 축소됐다. 지난주 확대됐던 상승폭이 이번주 줄어들며 서울 집값이 혼조세를 보이는 양상이다. 14일 한국부동산원이 발\n\n\n매일경제\n\n\n\n\n27\n개의 관련뉴스 더보기\n\n\n\n\n\n야놀자, 올 상반기 통합거래액 16.4조원 기록…\'역대 최대\'\n\n야놀자는 올해 상반기 통합거래액이 16조 4천억원을 기록했다고 14일 발표했다. 야놀자는 2025년 상반기 야놀자의

In [4]:
# 텍스트 분할
# 불러온 문서를 토큰 단위로 분할
# tiktoken 인코더를 사용하여 텍스트를 300자 단위로 분할, 청크간 50자의 오버랩을 둠
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300,
    chunk_overlap=50
)
splits = text_splitter.split_documents(docs)
splits

[Document(metadata={'source': 'https://news.naver.com/section/101'}, page_content='한국남부발전, 한화와 글로벌 LNG 협력 강화\n\n한국남부발전은 14일 서울 중구 더 플라자 호텔에서 한화에어로스페이스, 한화에너지와 ‘글로벌 LNG 협력 강화를 위한 Team KOREA 구축’ 업무협약을 체결했다고 밝혔다. 이번 협약은 최근 한미 관세협상 타결로'),
 Document(metadata={'source': 'https://news.naver.com/section/101'}, page_content='매일경제\n\n\n\n\n18\n개의 관련뉴스 더보기\n\n\n\n\n\n교보생명, 상반기 순익 5824억 전년비 5.4%↓…“미래 수익성 확보”'),
 Document(metadata={'source': 'https://news.naver.com/section/101'}, page_content='교보생명이 올해 상반기 5824억원의 순이익을 기록했다. 전년 대비 5.4% 감소한 수치로 보험영업이익이 감소했기 때문이다. 서울 광화문 소재 교보생명 본사 전경.(사진=교보생명) 14일 교보생명은 ‘2025년 상반\n\n\n이데일리\n\n\n\n\n13\n개의 관련뉴스 더보기'),
 Document(metadata={'source': 'https://news.naver.com/section/101'}, page_content='이데일리\n\n\n\n\n13\n개의 관련뉴스 더보기\n\n\n\n\n\n강릉·속초·경주·익산 ‘1주택자 세컨드홈 특례 지역’ 추가'),
 Document(metadata={'source': 'https://news.naver.com/section/101'}, page_content='정부가 1주택자가 비수도권에 집을 한 채 더 살 때 각종 세제 혜택을 받는 ‘세컨드홈’ 특례 대상 지역에 강릉·속초·익산·경주 등 9개 지역을 추가한다. 극심한 침체에서 벗어나지 못하고 있는 지방 건설

In [5]:
# 벡터 스토어 구축 및 문서 검색 설정
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=OpenAIEmbeddings()
)

vectorstore

In [6]:
# 최종적으로 상위 1개 문서만 반환 --> 'k'=1
# MMR 알고리즘이 고려할 관련서이 높은 문서 4개--> 'fetch_k'=4
retriever = vectorstore.as_retriever(
    search_type='mmr', # mmr 알고리즘
    search_kwargs={'k':1, 'fetch_k':4} # k : 반환할 문서의 개수, fetch_k : MMR 알고리즘이 고려할 문서의 수 설정

)
retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001F2730D6BD0>, search_type='mmr', search_kwargs={'k': 1, 'fetch_k': 4})

In [7]:
prompt = hub.pull('sungwoo/ragbasic')

In [9]:
llm = ChatOpenAI(model_name='gpt-4o-mini', temperature=0)
def format_docs(docs):
    formatted = '\n\n'.join(doc.page_content for doc in docs)
    return formatted

# 체인
rag_chain=(
    {'context':retriever | format_docs,
     'question' : RunnablePassthrough()}
     | prompt
     | llm
     | StrOutputParser()

)

rag_chain.invoke('국채 관련한 정보를 알려줘')
    

'국채에 대한 정보는 검색 결과에 포함되어 있지 않습니다. 따라서 국채에 대한 구체적인 내용을 제공할 수 없습니다. 추가적인 질문이 있으시면 말씀해 주세요.'

In [10]:
docs = retriever.invoke('국채 관련한 정보를 알려줘')
docs

[Document(metadata={'source': 'https://news.naver.com/section/101'}, page_content='정부가 1주택자가 비수도권에 집을 한 채 더 살 때 각종 세제 혜택을 받는 ‘세컨드홈’ 특례 대상 지역에 강릉·속초·익산·경주 등 9개 지역을 추가한다. 극심한 침체에서 벗어나지 못하고 있는 지방 건설 경기를 활성화\n\n\n동아일보')]

In [11]:
#  더 높은 다양성을 가진 문서 검색
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={'k':6, 'lambda_mult':0.2}
)

In [12]:
docs = retriever.invoke('국채 관련한 정보를 알려줘')
docs

[Document(metadata={'source': 'https://news.naver.com/section/101'}, page_content='정부가 1주택자가 비수도권에 집을 한 채 더 살 때 각종 세제 혜택을 받는 ‘세컨드홈’ 특례 대상 지역에 강릉·속초·익산·경주 등 9개 지역을 추가한다. 극심한 침체에서 벗어나지 못하고 있는 지방 건설 경기를 활성화\n\n\n동아일보'),
 Document(metadata={'source': 'https://news.naver.com/section/101'}, page_content='이데일리\n\n\n\n\n13\n개의 관련뉴스 더보기\n\n\n\n\n\n한국씨티은행, 상반기 당기순익 1831억…전년 대비 4.5% 증가'),
 Document(metadata={'source': 'https://news.naver.com/section/101'}, page_content='이재명 정부의 초대 금융당국 수장들이 임명된 가운데, 이번 인선에 따라 금융당국 조직개편이 어떻게 맞물려 진행될지 관심이 주목된다. 금융위원회와 금융감독원이 각각 새 수장을 맞으면서, 정권 교체 후부터 논의됐던 금융\n\n\n뉴시스\n\n51분전'),
 Document(metadata={'source': 'https://news.naver.com/section/101'}, page_content='산업통상자원부는 석유화학 구조개편에 대한 정부방침을 이달 중 발표하겠다고 14일 밝혔다. 현재 한국의 석유화학산업은 중국의 생산 능력 확대에 따른 대중 수출 감소, 글로벌 공급과잉에 위기를 겪고 있다. 이에 산업부는\n\n\n뉴스1'),
 Document(metadata={'source': 'https://news.naver.com/section/101'}, page_content='오는 25일로 예정된 한·미 정상회담에 재계 주요 총수들이 경제사절단으로 동행한다. 한국과 미국의 상호관세 최종안 발표를 앞두고 물밑 협상 지원에 나설 것으로 관측된다. 1

In [ ]:
# 50개의 문서를 고려하여 검색을 수행, 최종적으로 상위 5개의 문서만 반환 
# -> 보다 넓은 범위에서 검색을 수행한 후 가장 적합한 문서들만 선택하는데 유리
retriever = vectorstore.as_retriever(
    search_type='mmr', # mmr 알고리즘
    search_kwargs={'k':5, 'fetch_k':50}
)

In [15]:
docs = retriever.invoke('국채 관련한 정보를 알려줘')
docs

[Document(metadata={'source': 'https://news.naver.com/section/101'}, page_content='정부가 1주택자가 비수도권에 집을 한 채 더 살 때 각종 세제 혜택을 받는 ‘세컨드홈’ 특례 대상 지역에 강릉·속초·익산·경주 등 9개 지역을 추가한다. 극심한 침체에서 벗어나지 못하고 있는 지방 건설 경기를 활성화\n\n\n동아일보'),
 Document(metadata={'source': 'https://news.naver.com/section/101'}, page_content='한국일보\n\n18분전\n\n\n\n\n\n\n\n\n[오후초대석] 세제개편 논란에 박스권 갇힌 코스피…국내 증시 흐름은?'),
 Document(metadata={'source': 'https://news.naver.com/section/101'}, page_content='산업통상자원부는 석유화학 구조개편에 대한 정부방침을 이달 중 발표하겠다고 14일 밝혔다. 현재 한국의 석유화학산업은 중국의 생산 능력 확대에 따른 대중 수출 감소, 글로벌 공급과잉에 위기를 겪고 있다. 이에 산업부는\n\n\n뉴스1'),
 Document(metadata={'source': 'https://news.naver.com/section/101'}, page_content='한국씨티은행은 올해 상반기 전년 동기 대비 4.5% 증가한 1831억 원의 당기순이익을 올렸다고 14일 밝혔다. 상반기 총수익은 5595억 원으로, 전년도 상반기 대비 6.7% 감소했다. 2분기만 놓고 보면 순이익은\n\n\n뉴스1\n\n\n\n\n12\n개의 관련뉴스 더보기'),
 Document(metadata={'source': 'https://news.naver.com/section/101'}, page_content='국내에서 많이 팔리는 유명 정수기의 중국산 짝퉁 필터를 만들어 밀반입한 업체가 세관 당국에 적발됐습니다. 정품과 달리 성능도 크게 떨어져 염소와 납 성분을 제

In [ ]:
# 특정 임계값 이상의 유사도 점수를 가진 문서만 검색
# 유사도 점수가 0.8이상인 문서만 검색
retriever = vectorstore.as_retriever(
    search_type = 'similarity_score_threshold', # search_type -> 다른 검색 
    search_kwargs={'score_threshold':0.8}
)
# 가장 유사한 문서 하나만 검색
retriever = vectorstore.as_retriever(search_kwargs={'k':1})


In [17]:
docs = retriever.invoke('국채 관련한 정보를 알려줘')
docs

[Document(metadata={'source': 'https://news.naver.com/section/101'}, page_content='정부가 1주택자가 비수도권에 집을 한 채 더 살 때 각종 세제 혜택을 받는 ‘세컨드홈’ 특례 대상 지역에 강릉·속초·익산·경주 등 9개 지역을 추가한다. 극심한 침체에서 벗어나지 못하고 있는 지방 건설 경기를 활성화\n\n\n동아일보')]

In [ ]:
# Multiquery + Unique-union

# 문서로드
loader = WebBaseLoader(
    web_paths=('https://news.naver.com/section/101',),
    bs_kwargs = dict(
        parse_only=bs4.SoupStrainer(
            class_ = ("sa_text", "sa_item_SECTION_HEADLINE")
        )
    ),
)
docs = loader.load() 

# 텍스트 분할
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300,
    chunk_overlap=50,
)
splits = text_splitter.split_documents(docs)

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=OpenAIEmbeddings()
)
retriever = vectorstore.as_retriever()


MissingSchema: Invalid URL 'h': No scheme supplied. Perhaps you meant https://h?

In [21]:
# Multi Query 기법을 위한 템플릿
from langchain.prompts import ChatPromptTemplate

template = """
당신은 AI 언어 모델 조수입니다. 당신의 임무는 주어진 사용자 질문에 대해 벡터 데이터베이스에서 관련 문서를 검색할 수 있도록 다섯 가지 다른 버전을 생성하는 것입니다. 
사용자 질문에 대한 여러 관점을 생성함으로써, 거리 기반 유사성 검색의 한계를 극복하는 데 도움을 주는 것이 목표입니다. 
각 질문은 새 줄로 구분하여 제공하세요. 원본 질문: {question}
"""

In [23]:
prompt_perspectives = ChatPromptTemplate.from_template(template)
generate_queries=(
    prompt_perspectives
    | ChatOpenAI(model_name='gpt-4o-mini', temperature=0)
    | StrOutputParser()
    | (lambda x:x.split('\n'))
)

generated_query = generate_queries.invoke('집값의 향방?')
generated_query

['집값의 미래 전망은 어떻게 될까요?  ',
 '현재 집값의 추세와 앞으로의 변화는 어떤 영향을 받을까요?  ',
 '부동산 시장에서 집값이 오를지 내릴지에 대한 예측은 무엇인가요?  ',
 '경제적 요인들이 집값에 미치는 영향은 어떤 것들이 있을까요?  ',
 '향후 몇 년간 집값의 변동성을 어떻게 분석할 수 있을까요?  ']

In [24]:
# 함수 정의
from langchain.load import dumps, loads

# 매개변수의 자료형에 대한 힌트 -> 중첩리스트
def get_unique_union(documents: list[list]):
    """고유한 문서들의 합집합(중복제거)을 생성하는 함수"""
    
    # flattened_docs = []
    # for sublist in documents:
    #     for doc in sublist:
    #         flattened_docs.append(dumps(doc))
    # 위의 코드를 한줄로 

    # 평탄화(중첩된 리스트를 단일 리스트로), 직렬화(문자열로)
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist] # 중첩리스트

    # 중복된 문서를 제거하고(set을 사용해서) 고유한 문서만 남긴다
    unique_docs = list(set(flattened_docs)) # set() : 세트자료형으로 형변환, 세트자료형은 중복제거됨

    # 고유한 문서를 원래의 문서 객체로 변환하여 반환
    return [loads(doc) for doc in unique_docs]

In [25]:
question = '집값의 향방?'

# 문서 검색 체인 구성
# retriever.map() -> 관련 문서들을 검색(개별적으로 문서 검색 수행 -> 결과로 문서들의 리스트 반환)
retriever_chain = generate_queries | retriever.map() | get_unique_union

# 실제 문서 검색 -> 고유한 문서(중복없는) 반환
docs = retriever_chain.invoke({'question':question})

# 검색된 고유 문서들의 개수와 문서를 출력
print(len(docs))
docs

3


[Document(metadata={'source': 'https://news.naver.com/section/101'}, page_content='앞으로 강원도 강릉이나 속초에 두 번째 집을 사도 1주택자로 간주돼 세금 부담을 덜 수 있게 된다. 대규모 사회간접자본(SOC) 사업을 추진할 때 거쳐야 하는 예비타당성조사(예타) 대상 기준도 기존보다 두 배 많은 \n\n\n한국일보\n\n18분전'),
 Document(metadata={'source': 'https://news.naver.com/section/101'}, page_content='천정부지로 뛰고 있는 금값에 이어 은값도 역시 치솟고 있습니다. 공급도 얼마 없는데 산업재 수요가 증가했기 때문입니다. 시중 유동성 공급이 이어지면서 은 값은 더 오를 것이란 전망도 나옵니다. 엄하은 기자의 보도입니\n\n\nSBS Biz\n\n58분전'),
 Document(metadata={'source': 'https://news.naver.com/section/101'}, page_content='부동산원 주간 아파트 동향 서울 0.14%→0.10% 축소 서울 아파트값 상승폭이 일주일 사이 다시 축소됐다. 지난주 확대됐던 상승폭이 이번주 줄어들며 서울 집값이 혼조세를 보이는 양상이다. 14일 한국부동산원이 발\n\n\n매일경제\n\n\n\n\n27\n개의 관련뉴스 더보기')]

In [26]:
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough

# RAG
template = '''다음 맥락을 바탕으로 질문에 답변하세요:
{context}

질문: {question}
'''

prompt = ChatPromptTemplate.from_template(template)

llm = ChatOpenAI(model_name='gpt-4o-mini', temperature=0)

final_rag_chain = (
    {'context': retriever_chain,
     'question': RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke(question)

'최근 서울 아파트값의 상승폭이 줄어들며 혼조세를 보이고 있다는 보도가 있습니다. 부동산원에 따르면, 서울의 아파트값 상승률이 지난주 0.14%에서 이번 주 0.10%로 축소되었습니다. 이는 집값이 상승세에서 다소 주춤하고 있음을 나타냅니다. 따라서 현재 집값의 향방은 상승세가 둔화되고 있는 것으로 해석할 수 있습니다.'

In [27]:
import os
from dotenv import load_dotenv

load_dotenv()
openai_api_key = os.getenv('OPENAI_API_KEY')

# OpenAI API key가 없으면 예외를 발생시킨다
if not openai_api_key:
    raise ValueError('OpenAI API 키가 없습니다. 한번 더 확인해주세요!')


In [28]:
# 라이브러리 불러오기
import bs4
from langchain import hub
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.prompts import ChatPromptTemplate

#### INDEXING ####

# 문서 로드
loader = WebBaseLoader(
    web_paths=("https://news.naver.com/section/101",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("sa_text", "sa_item_SECTION_HEADLINE")
        )
    ),
)
docs = loader.load()

# 텍스트 분할
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300,
    chunk_overlap=50
)

splits = text_splitter.split_documents(docs)

# Index
vectorstore = Chroma.from_documents(
    documents=splits, 
    embedding=OpenAIEmbeddings()
)
retriever = vectorstore.as_retriever()

In [29]:
# RAG-Fusion: 관련 검색 쿼리 생성

# template = """You are a helpful assistant that generates multiple search queries based on a single input query. \n
# Generate multiple search queries related to: {question} \n
# Output (4 queries):"""

template = '''당신은 주어진 하나의 질문을 기반으로 여러 검색 쿼리를 생성하는 유용한 조수입니다. \n
다음 질문과 관련된 여러 검색 쿼리를 생성하세요: {question} \n
출력 (4개의 쿼리):'''

prompt_rag_fusion = ChatPromptTemplate.from_template(template)

In [30]:
generate_queries = (
    prompt_rag_fusion
    | ChatOpenAI(temperature=0)
    | StrOutputParser()
    | (lambda x: x.split('\n'))
)

In [31]:
# RRF 알고리즘 구현
from langchain.load import dumps, loads

def reciprocal_rank_fusion(results: list[list], k=60, top_n=2):
    """"
    RRF 알고리즘을 이용한 함수 정의
    여러개의 순위가 매겨진 문서 리스트를 받아, RRF 공식을 사용하여 문서의 최종순위를
    계산하는 함수
    k -> 상수 개념(보통 60을 사용)
    top_n -> 반환할 우선 순위가 높은 문서의 개수
    """

    # 각 고유한 문서에 대한 점수 저장
    fused_scores={}

    # 순위가 매겨진 문서리스트(results)를 순회
    for docs in results:
        # for 인덱스번호, 값 in enumerate(리스트명):
        for rank, doc in enumerate(docs):
            # 문서를 문자열 형식으로  직렬화하여 딕셔너리의 키로 사용
            doc_str = dumps(doc)

            # 해당 문서가 아직 딕셔너리에 없으면 초기 점수를 0으로 추가
            if doc_str not in fused_scores:
                # fused_scores = {'doc_str':0}
                fused_scores[doc_str] = 0 # 딕셔너리 fused_scores에 doc_str 키의 값0을 추가
            
            # 문서의 현재 점수를 가져온다(이전에 계산된 점수)
            previous_score = fused_scores[doc_str]

            # RRF 공식을 사용하여 문서의 점수를 업데이트 --> 1 / (k + 순위)
            fused_scores[doc_str] += 1 / (k + rank)
    
    # 문서들을 계산된 점수에 따라 내림차순으로 정렬(reverse=True)하여 최종적으로 재정렬된 결과를 얻는다.
    # 딕셔너리이름.items() --> [(키1, 값1), (키2, 값2)] --> 키와 값(쌍)을 튜플 형태로 묶어 리스트
    # lambda x: x[1] -> (0, 1) 인덱스 0번은 키, 인덱스 1번은 값 값으로 내림차순  
    reranked_results=[
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]
    # 재정렬된 결과에서 우선순위가 높은 top_n개의 문서만 반환
    return reranked_results[:top_n]

In [32]:
# RAG-Fusion 체인을 구성한다.
retrieval_chain_rag_fusion = generate_queries | retriever.map() | reciprocal_rank_fusion

# 체인을 실행하여 질문에 대한 검색된 문서들을 가져온다.
question = '향후 집값에 대해서 알려줘'
docs = retrieval_chain_rag_fusion.invoke({'question': question})

# 검색된 고유 문서들의 개수를 출력한다.
print(len(docs))

# 검색된 고유 문서들을 출력한다.
docs

2


[(Document(metadata={'source': 'https://news.naver.com/section/101'}, page_content="서울 아파트값 상승폭이 전 권역에서 일제히 축소됐다. 6·27 대출 규제 이후 한동안 완만한 둔화세를 이어가다 지난주 반짝 반등했지만, 한 주 만에 다시 힘이 빠진 모습이다. 14일 한국부동산원이 발표한 '2025년\n\n\n파이낸셜뉴스\n\n46분전"),
  0.06480614103564923),
 (Document(metadata={'source': 'https://news.naver.com/section/101'}, page_content="조세일보\n\n22분전\n\n\n\n\n\n\n\n\n광역시마저 동네 절반 '빈집'…'지방 부동산 살리기' 대책은?"),
  0.03333333333333333)]

In [33]:
# RAG
template = '''다음 맥락을 바탕으로 질문에 답변하세요:

{context}

질문: {question}
'''

prompt = ChatPromptTemplate.from_template(template)

llm = ChatOpenAI(model_name='gpt-4o-mini', temperature=0)

final_rag_chain = (
    {'context': retrieval_chain_rag_fusion,
     'question': RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke(question)

'제공된 맥락에서는 강원도 강릉이나 속초에 두 번째 집을 사도 1주택자로 간주되어 세금 부담이 덜어진다는 내용이 있습니다. 이는 주택 구매를 장려할 수 있는 정책으로, 수요 증가로 이어질 가능성이 있습니다. 또한, 대규모 사회간접자본(SOC) 사업의 추진이 언급되었는데, 이러한 사업은 지역 경제 활성화에 기여할 수 있어 집값에 긍정적인 영향을 미칠 수 있습니다.\n\n반면, "광역시마저 동네 절반 \'빈집\'"이라는 언급은 지방 부동산 시장의 어려움을 나타내며, 이는 특정 지역의 집값 하락 요인으로 작용할 수 있습니다. 따라서 향후 집값은 지역별로 상이할 수 있으며, 정책 변화와 경제 상황에 따라 영향을 받을 것으로 보입니다.'